In [1]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import cv2
import random
import concurrent.futures




In [2]:
train_image_path = "../input/plant-pathology-2021-fgvc8/train_images/"
train_file = "../input/plant-pathology-2021-fgvc8/train.csv"
submission_file = "../working/submission.csv"
test_image_path = "../input/plant-pathology-2021-fgvc8/test_images/"

if os.path.exists(submission_file):
    os.remove(submission_file)




In [3]:
def mean_green(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    return img[:, :, 1].mean()  # green channel


train_df = pd.read_csv(train_file)

complex_idxs = train_df[train_df["labels"].str.contains("complex")].index.tolist()
healthy_idxs = train_df[~train_df["labels"].str.contains("complex")].index.tolist()


def greens_of_indices(idxs):
    """Compute mean green values for a list of dataframe indices using parallel I/O."""
    # Build full image paths preserving order
    paths = [os.path.join(train_image_path, train_df.iloc[i]["image"]) for i in idxs]

    greens = []
    # Use a thread pool (I/O bound work) while keeping order via map
    with concurrent.futures.ThreadPoolExecutor(
        max_workers=min(32, os.cpu_count() or 1)
    ) as executor:
        for g in tqdm(
            executor.map(mean_green, paths),
            total=len(paths),
            desc="Calibrating",
        ):
            if g is not None:
                greens.append(g)
    return greens


greens_complex = greens_of_indices(complex_idxs)
greens_healthy = greens_of_indices(healthy_idxs)

default_threshold = 100
if greens_complex and greens_healthy:
    median_complex = np.median(greens_complex)
    median_healthy = np.median(greens_healthy)
    calibrated_threshold = (median_complex + median_healthy) / 2
else:
    calibrated_threshold = default_threshold

print(f"Calibrated green threshold (median based): {calibrated_threshold:.2f}")




Calibrating: 100%|██████████| 13194/13194 [02:53<00:00, 76.17it/s]


Calibrated green threshold (median based): 163.26


In [4]:
def predict_label(image_path, threshold):
    """Heuristic: low average green → 'complex', else 'healthy'."""
    img = cv2.imread(image_path)
    if img is None:
        return "healthy"  # fallback
    mean_green_val = img[:, :, 1].mean()
    return "complex" if mean_green_val < threshold else "healthy"


test_images = [
    f
    for f in os.listdir(test_image_path)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

# Parallel prediction while preserving order
test_paths = [os.path.join(test_image_path, name) for name in test_images]

pred_labels = []
with concurrent.futures.ThreadPoolExecutor(
    max_workers=min(32, os.cpu_count() or 1)
) as executor:
    for label in tqdm(
        executor.map(lambda p: predict_label(p, calibrated_threshold), test_paths),
        total=len(test_paths),
        desc="Predicting labels",
    ):
        pred_labels.append(label)

sub = pd.DataFrame({"image": test_images, "labels": pred_labels})
print(sub.head())
print(f"Total predictions: {len(sub)}")




Predicting labels: 100%|██████████| 3727/3727 [01:25<00:00, 43.79it/s]

                  image   labels
0  df98c83c4d383c2d.jpg  healthy
1  817e97dad0c33ae0.jpg  complex
2  e984c49aef3d54a1.jpg  healthy
3  8cf2ee38c423d837.jpg  healthy
4  e5319c4b1ce90d5e.jpg  complex
Total predictions: 3727


In [5]:
sub.to_csv(submission_file, index=False)
print(f"Submission saved to {submission_file}")

Submission saved to ../working/submission.csv
